## BÀI 1

In [1]:
import sqlite3
import pandas as pd
from datetime import datetime, timedelta

conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# Tạo bảng course
cursor.execute('''
CREATE TABLE course (
    id INTEGER PRIMARY KEY,
    course_name TEXT
)
''')

courses = [(12, 'Giai tich'), (34, 'Thong ke'), (26, 'Tin hoc')]
cursor.executemany('INSERT INTO course VALUES (?, ?)', courses)

# Tạo bảng student
cursor.execute('''
CREATE TABLE student (
    student_id INTEGER,
    name TEXT,
    class TEXT,
    course_id INTEGER,
    score REAL
)
''')

students = [
    (1, 'Nguyen Minh Hoang', 'May Tinh', 12, 6.7),
    (2, 'Tran Thi Lan', 'Kinh Te', 34, 9.2),
    (3, 'Pham Van Nam', 'Toan Tin', None, 7.9),
    (4, 'Le Thanh Huyen', 'Toan Tin', 20, 7.2),
    (5, 'Vu Quoc Anh', 'May Tinh', 24, 8.0),
    (6, 'Dang Thuy Linh', 'May Tinh', 24, 5.5),
    (7, 'Bui Tien Dung', 'Kinh Te', 34, 9.2),
    (8, 'Ho Ngoc Mai', 'Toan Tin', 20, 8.8),
    (9, 'Duong Huu Phuc', 'Kinh Te', None, 7.2),
    (10, 'Cao Thi Hanh', 'May Tinh', None, 7.0)
]
cursor.executemany('INSERT INTO student VALUES (?, ?, ?, ?, ?)', students)
conn.commit()

c:\Users\OS\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


In [3]:
# Tích Descartes
df_cartesian = pd.read_sql_query('''
    SELECT * FROM student, course
''', conn)
print("Tích Descartes:\n", df_cartesian.head())

# INNER JOIN
df_inner = pd.read_sql_query('''
    SELECT s.*, c.course_name
    FROM student s
    INNER JOIN course c ON s.course_id = c.id
''', conn)
print("INNER JOIN:\n", df_inner)

# LEFT JOIN
df_left = pd.read_sql_query('''
    SELECT s.*, c.course_name
    FROM student s
    LEFT JOIN course c ON s.course_id = c.id
''', conn)
print("LEFT JOIN:\n", df_left)

# RIGHT JOIN 
df_right = pd.read_sql_query('''
    SELECT s.*, c.course_name
    FROM course c
    LEFT JOIN student s ON s.course_id = c.id
''', conn)
print("RIGHT JOIN (mô phỏng):\n", df_right)

# FULL OUTER JOIN 
df_full = pd.read_sql_query('''
    SELECT s.*, c.course_name
    FROM student s
    LEFT JOIN course c ON s.course_id = c.id
    UNION
    SELECT s.*, c.course_name
    FROM student s
    RIGHT JOIN course c ON s.course_id = c.id
''', conn)
print("FULL OUTER JOIN:\n", df_full)

Tích Descartes:
    student_id               name     class  course_id  score  id course_name
0           1  Nguyen Minh Hoang  May Tinh       12.0    6.7  12   Giai tich
1           1  Nguyen Minh Hoang  May Tinh       12.0    6.7  26     Tin hoc
2           1  Nguyen Minh Hoang  May Tinh       12.0    6.7  34    Thong ke
3           2       Tran Thi Lan   Kinh Te       34.0    9.2  12   Giai tich
4           2       Tran Thi Lan   Kinh Te       34.0    9.2  26     Tin hoc
INNER JOIN:
    student_id               name     class  course_id  score course_name
0           1  Nguyen Minh Hoang  May Tinh         12    6.7   Giai tich
1           2       Tran Thi Lan   Kinh Te         34    9.2    Thong ke
2           7      Bui Tien Dung   Kinh Te         34    9.2    Thong ke
LEFT JOIN:
    student_id               name     class  course_id  score course_name
0           1  Nguyen Minh Hoang  May Tinh       12.0    6.7   Giai tich
1           2       Tran Thi Lan   Kinh Te       34.0    9

## BÀI 2

In [4]:
# Xóa sinh viên có course_id không tồn tại trong course
cursor.execute('''
    DELETE FROM student
    WHERE course_id IS NOT NULL AND course_id NOT IN (SELECT id FROM course)
''')
conn.commit()

# Cập nhật course_id còn thiếu bằng một môn học có trong course 
cursor.execute('''
    UPDATE student
    SET course_id = (SELECT id FROM course LIMIT 1)
    WHERE course_id IS NULL
''')
conn.commit()

df_updated = pd.read_sql_query('SELECT * FROM student', conn)
print("Sau cập nhật:\n", df_updated)

Sau cập nhật:
    student_id               name     class  course_id  score
0           1  Nguyen Minh Hoang  May Tinh         12    6.7
1           2       Tran Thi Lan   Kinh Te         34    9.2
2           3       Pham Van Nam  Toan Tin         12    7.9
3           7      Bui Tien Dung   Kinh Te         34    9.2
4           9     Duong Huu Phuc   Kinh Te         12    7.2
5          10       Cao Thi Hanh  May Tinh         12    7.0


In [5]:
# tổng số sinh viên, điểm trung bình của từng lớp
df_class = pd.read_sql_query('''
    SELECT class,
           COUNT(student_id) AS total_students,
           AVG(score) AS avg_score
    FROM student
    GROUP BY class
''', conn)
print(df_class)

      class  total_students  avg_score
0   Kinh Te               3   8.533333
1  May Tinh               2   6.850000
2  Toan Tin               1   7.900000


In [6]:
# tổng số sinh viên, điểm trung bình của từng môn 
df_course_stats = pd.read_sql_query('''
    SELECT c.course_name,
           COUNT(s.student_id) AS total_students,
           AVG(s.score) AS avg_score
    FROM student s
    JOIN course c ON s.course_id = c.id
    GROUP BY c.course_name
''', conn)
print(df_course_stats)

  course_name  total_students  avg_score
0   Giai tich               4        7.2
1    Thong ke               2        9.2


In [7]:
# phân loại thi đua theo điểm trung bình môn học 
def classify(avg):
    if avg >= 9.0:
        return 'Xuất sắc'
    elif avg >= 6.0:
        return 'Tốt'
    else:
        return 'Kém'

df_course_stats['classification'] = df_course_stats['avg_score'].apply(classify)
print(df_course_stats)

  course_name  total_students  avg_score classification
0   Giai tich               4        7.2            Tốt
1    Thong ke               2        9.2       Xuất sắc


## BÀI 3

In [11]:
# xếp hạng sinh viên thông qua điểm số
df_rank_all = pd.read_sql_query('''
    SELECT student_id, name, score,
           ROW_NUMBER() OVER (ORDER BY score DESC) AS rank_high_to_low,
           ROW_NUMBER() OVER (ORDER BY score ASC) AS rank_low_to_high
    FROM student
''', conn)
print(df_rank_all)

   student_id               name  score  rank_high_to_low  rank_low_to_high
0           2       Tran Thi Lan    9.2                 1                 5
1           7      Bui Tien Dung    9.2                 2                 6
2           3       Pham Van Nam    7.9                 3                 4
3           9     Duong Huu Phuc    7.2                 4                 3
4          10       Cao Thi Hanh    7.0                 5                 2
5           1  Nguyen Minh Hoang    6.7                 6                 1


In [12]:
# top 3 cao nhất và thấp nhất 
top3_high = df_rank_all.nsmallest(3, 'rank_high_to_low')[['student_id', 'name', 'score']]
top3_low = df_rank_all.nsmallest(3, 'rank_low_to_high')[['student_id', 'name', 'score']]
print("Top 3 cao nhất:\n", top3_high)
print("Top 3 thấp nhất:\n", top3_low)

Top 3 cao nhất:
    student_id           name  score
0           2   Tran Thi Lan    9.2
1           7  Bui Tien Dung    9.2
2           3   Pham Van Nam    7.9
Top 3 thấp nhất:
    student_id               name  score
5           1  Nguyen Minh Hoang    6.7
4          10       Cao Thi Hanh    7.0
3           9     Duong Huu Phuc    7.2


In [13]:
# xếp hạng sinh viên thông qua điểm số theo lớp học 
df_rank_class = pd.read_sql_query('''
    SELECT student_id, name, class, score,
           ROW_NUMBER() OVER (PARTITION BY class ORDER BY score DESC) AS rank_in_class
    FROM student
''', conn)
print(df_rank_class)

   student_id               name     class  score  rank_in_class
0           2       Tran Thi Lan   Kinh Te    9.2              1
1           7      Bui Tien Dung   Kinh Te    9.2              2
2           9     Duong Huu Phuc   Kinh Te    7.2              3
3          10       Cao Thi Hanh  May Tinh    7.0              1
4           1  Nguyen Minh Hoang  May Tinh    6.7              2
5           3       Pham Van Nam  Toan Tin    7.9              1


In [36]:
# Top 3 cao nhất và thấp nhất theo lớp học
for class_name in df_rank_class['class'].unique():
    df_c = df_rank_class[df_rank_class['class'] == class_name]
    print(f"\nLớp {class_name}:")
    print(f"  Top 3 cao nhất:\n{df_c.head(3)[['name', 'score']]}")
    print(f"  Top 3 thấp nhất:\n{df_c.tail(3)[['name', 'score']]}")


Lớp Kinh Te:
  Top 3 cao nhất:
             name  score
0    Tran Thi Lan    9.2
1   Bui Tien Dung    9.2
2  Duong Huu Phuc    7.2
  Top 3 thấp nhất:
             name  score
0    Tran Thi Lan    9.2
1   Bui Tien Dung    9.2
2  Duong Huu Phuc    7.2

Lớp May Tinh:
  Top 3 cao nhất:
                name  score
3       Cao Thi Hanh    7.0
4  Nguyen Minh Hoang    6.7
  Top 3 thấp nhất:
                name  score
3       Cao Thi Hanh    7.0
4  Nguyen Minh Hoang    6.7

Lớp Toan Tin:
  Top 3 cao nhất:
           name  score
5  Pham Van Nam    7.9
  Top 3 thấp nhất:
           name  score
5  Pham Van Nam    7.9


In [16]:
# xếp hạng sinh viên thông qua điểm số theo mã môn học 
df_rank_course = pd.read_sql_query('''
    SELECT s.student_id, s.name, c.course_name, s.score,
           ROW_NUMBER() OVER (PARTITION BY c.course_name ORDER BY s.score DESC) AS rank_in_course
    FROM student s
    JOIN course c ON s.course_id = c.id
''', conn)
print(df_rank_course)

   student_id               name course_name  score  rank_in_course
0           3       Pham Van Nam   Giai tich    7.9               1
1           9     Duong Huu Phuc   Giai tich    7.2               2
2          10       Cao Thi Hanh   Giai tich    7.0               3
3           1  Nguyen Minh Hoang   Giai tich    6.7               4
4           2       Tran Thi Lan    Thong ke    9.2               1
5           7      Bui Tien Dung    Thong ke    9.2               2


In [35]:
# top 3 cao nhất và thấp nhất theo mã môn học
for course in df_rank_course['course_name'].unique():
    df_c = df_rank_course[df_rank_course['course_name'] == course]
    print(f"\n{course}:")
    print(f"  Top 3 cao nhất:\n{df_c.head(3)[['name', 'score']]}")
    print(f"  Top 3 thấp nhất:\n{df_c.tail(3)[['name', 'score']]}")


Giai tich:
  Top 3 cao nhất:
             name  score
0    Pham Van Nam    7.9
1  Duong Huu Phuc    7.2
2    Cao Thi Hanh    7.0
  Top 3 thấp nhất:
                name  score
1     Duong Huu Phuc    7.2
2       Cao Thi Hanh    7.0
3  Nguyen Minh Hoang    6.7

Thong ke:
  Top 3 cao nhất:
            name  score
4   Tran Thi Lan    9.2
5  Bui Tien Dung    9.2
  Top 3 thấp nhất:
            name  score
4   Tran Thi Lan    9.2
5  Bui Tien Dung    9.2


## BÀI 4

In [33]:
# xác định thời gian tốt nghiệp
from datetime import datetime, timedelta

# Thêm cột
try:
    cursor.execute('ALTER TABLE student ADD COLUMN graduation_date DATETIME')
except:
    pass

# Xếp hạng (cao nhất = hạng 1)
df_rank = pd.read_sql_query('SELECT student_id, ROW_NUMBER() OVER (ORDER BY score DESC) AS rank FROM student', conn)

# Cập nhật: hiện tại + số hạng (tháng)
now = datetime.now()
for _, row in df_rank.iterrows():
    grad_date = now + timedelta(days=int(row['rank']) * 30)
    cursor.execute('UPDATE student SET graduation_date = ? WHERE student_id = ?', (grad_date, row['student_id']))

conn.commit()

# Kết quả
pd.read_sql_query('SELECT student_id, name, score, graduation_date FROM student ORDER BY score DESC', conn)

C:\Users\OS\AppData\Local\Temp\ipykernel_8288\41645129.py:17: DeprecationWarning: The default datetime adapter is deprecated as of Python 3.12; see the sqlite3 documentation for suggested replacement recipes
  cursor.execute('UPDATE student SET graduation_date = ? WHERE student_id = ?', (grad_date, row['student_id']))


,student_id,name,score,graduation_date
0,2,Tran Thi Lan,9.2,2026-05-03 09:01:03.756951
1,7,Bui Tien Dung,9.2,2026-06-02 09:01:03.756951
2,3,Pham Van Nam,7.9,2026-07-02 09:01:03.756951
3,9,Duong Huu Phuc,7.2,2026-08-01 09:01:03.756951
4,10,Cao Thi Hanh,7.0,2026-08-31 09:01:03.756951
5,1,Nguyen Minh Hoang,6.7,2026-09-30 09:01:03.756951
